# Frontend route experiment — keputusan kandidat saat ini

CPU frontend + encoder penuh CPU+GPU lolos pada kedua lagu. GPU frontend 30 detik mempertahankan label pada uji satu lagu, tetapi gagal batas probabilitas. Untuk tahap berikutnya, lanjutkan [encoder_weight_packaging.ipynb](encoder_weight_packaging.ipynb). Tidak perlu mengulang eksperimen GPU frontend untuk kandidat ini.

# Kecepatan convolution frontend: CPU vs GPU frontend 30 detik

Graph GLU repair telah menyelesaikan enam run iPhone dan full validation **32 lagu / 451 section** dengan label sama seperti pipeline palette6 sebelumnya (akurasi 79,16%, macro F1 0,767214). Perbaikan tidak menambah penurunan label. Selisih satu pre-chorus terhadap fitur training asli berasal dari pipeline sebelumnya dan tetap dilaporkan.

Batch iPhone memperlihatkan dua convolution frontend memakai sekitar 41% waktu PCM pipeline. Uji ini mengubah **compute route dua frontend convolution pada window penuh 30 detik saja**, menggunakan compiled model dan bobot yang sama. Frontend window 120 detik dan partial selalu CPU; log yang dikirim berhenti pada panggilan frontend GPU pertama untuk 120 detik. Penyebab terminasi belum diketahui karena tidak ada alasan crash eksplisit. Encoder window penuh tetap CPU+GPU graph repair; log-mel, partial transformer, label head dan konteks tetap CPU. Tidak ada training, kuantisasi atau pemotongan window baru.

Project **Frontend 30s** terpisah dari SongMap dan GLU Repair. Kebijakan GPU frontend dibatasi ke input mel [1,128,3000] pada window penuh 30 detik. Instance convolution diganti saat shape/mode berubah. Ini kandidat mitigasi yang masih perlu diuji di perangkat. Persiapan tetap typecheck Swift terhadap SDK iPhone dan memeriksa pipeline PCM CPU Mac sebelum membuka Xcode. Ini bukan pengaktifan GPU di aplikasi produksi.

Untuk benchmark compute route ini, preflight menggunakan `functional-benchmark`: semua input harus finite dengan shape yang sama, label akhir harus identik, dan probabilitas base/akhir harus berbeda paling banyak `1e-4`. Toleransi feature tetap `atol=rtol=1e-4`; pelanggarannya disimpan sebagai `numerical_passed=false` dan dipecah per stream. Proyek dapat dibuat jika pemeriksaan output lolos, sementara feature parity tetap dilaporkan gagal. Ini perubahan kebijakan penerimaan benchmark, bukan klaim kesetaraan numerik embedding atau persetujuan produksi.


In [ ]:
from pathlib import Path
from datetime import datetime
import json,hashlib,subprocess

ROOT=Path.cwd().resolve()
assert (ROOT/'scripts/prepare_encoder_pipeline_iphone.py').is_file(), 'Buka dari root SongMap'
SONGSTRUCT_ROOT=ROOT.parent/'Section-Annotations'
RUN_PYTHON=SONGSTRUCT_ROOT/'.venv-songformer311/bin/python'  # Pertahankan executable venv; jangan resolve symlink.
assert RUN_PYTHON.is_file(), f'Python venv tidak ditemukan: {RUN_PYTHON}'
def sha(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()

# Identitas report full validation GLU repair yang baru selesai dan telah ditinjau.
EXPECTED_SELECTION='f65edb95bae3fb2a948d742c647168763217180fb67b68ab5d82ad9ca5fc491d'
EXPECTED_REPAIR='3e84fe326e03a8f0e0eed1bff07c1fe00ea15cbf7ed3175f70c8fa9e14e408c0'
candidates=[]
for path in (ROOT/'outputs.local').glob('encoder-glu-quality-*/validation-all.json'):
    r=json.loads(path.read_text())
    if (r.get('selection_identity')==EXPECTED_SELECTION and r.get('graph_repair_sha256')==EXPECTED_REPAIR
            and r.get('full_validation') and r.get('section_agreement')==1
            and r.get('comparison_to_current_pipeline',{}).get('no_observed_drop')):candidates.append(path)
assert candidates, 'Full validation GLU repair yang sesuai belum ditemukan; selesaikan encoder_glu_quality.ipynb'
QUALITY=max(candidates,key=lambda p:p.stat().st_mtime).resolve()
REPAIR=next((p.resolve() for p in (ROOT/'outputs.local').glob('encoder-glu-repair-*/repair-report.json') if sha(p)==EXPECTED_REPAIR),None)
assert REPAIR, 'Repair yang sama dengan full validation belum ditemukan'
OUTPUT=ROOT/'outputs.local'/('frontend-short-routes-'+datetime.now().strftime('%Y%m%d-%H%M%S-%f'))
print('Quality:',QUALITY)
print('Repair:',REPAIR)
print('Project baru:',OUTPUT)

def run_logged(args,log):
    log=Path(log);log.parent.mkdir(parents=True,exist_ok=True)
    with log.open('w',encoding='utf-8') as handle:
        process=subprocess.Popen([str(x) for x in args],cwd=ROOT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in process.stdout:
                print(line,end='');handle.write(line);handle.flush()
            status=process.wait()
        except KeyboardInterrupt:
            process.terminate();process.wait();raise
    if status:raise RuntimeError(f'Persiapan gagal ({status}); lihat {log}. Jangan lanjut build sebelum preflight selesai. Gunakan OUTPUT baru untuk percobaan ulang.')


In [ ]:
run_logged([RUN_PYTHON,'-u',ROOT/'scripts/prepare_encoder_pipeline_iphone.py',
    '--songstruct-root',SONGSTRUCT_ROOT,'--songmap-root',ROOT,
    '--quality-report',QUALITY,'--graph-repair',REPAIR,
    '--benchmark-frontend-routes','--preflight-policy','functional-benchmark','--output-dir',OUTPUT],Path(str(OUTPUT)+'.prepare.log'))
proof=json.loads((OUTPUT/'preparation.json').read_text())
assert proof['frontend_route_probe'] and proof['native_typecheck_passed'] and proof['mac_pcm_pipeline_passed']
assert proof['mac_pcm_functional_passed'] and proof['mac_preflight_policy']=='functional-benchmark'
print('Feature parity CPU Mac:',proof['mac_pcm_feature_parity_passed'])
for check in proof['checks']:
    print(check['track_id'],'numerical:',check['numerical_passed'],'functional:',check['functional_passed'],check['native_context'])
assert proof['frontend_gpu_policy']=='full-30s-only-v1'
assert proof['graph_repair_sha256']==EXPECTED_REPAIR and proof['quality_sha256']==sha(QUALITY)
print('Project siap:',OUTPUT/'Benchmark.xcodeproj')
subprocess.run(['open',str(OUTPUT/'Benchmark.xcodeproj')],check=True)


## Uji pada iPhone

Pilih Signing Team dan iPhone fisik. Tidak perlu launch argument. Gunakan aplikasi baru **Frontend 30s**, dengan perangkat tidak panas dan Low Power Mode mati.

1. Pada **Convolution frontend**, pilih **GPU 30s / CPU others**. Pilih lagu pertama lalu **Run selected track**.
2. Periksa `label_parity_passed=true`, `all_predictions_finite=true`, dan `section_agreement_with_mac=1`. Jika gagal/crash, berhenti dan kirim hasil atau **Share last recorded stage** sebelum run baru.
3. Jika run tunggal lolos, tutup aplikasi lalu buka kembali. Tekan **Compare frontend routes (AB / BA)**. Ini menjalankan dua lagu, masing-masing dengan frontend CPU → GPU 30s → GPU 30s → CPU, sebanyak delapan run dalam satu proses. Pada mode GPU 30s, semua frontend 120 detik dan partial tetap CPU. Encoder penuh memakai CPU+GPU yang sama untuk seluruh run. Urutan tersebut membantu menilai pengaruh pemanasan/cache; bukan jaminan cold cache atau kondisi thermal identik.
4. Kirim **batch JSON** dan `preparation.json`. Batch berhenti otomatis pada selisih label atau probabilitas base/akhir >`1e-4`. Jika crash native, buka ulang dan bagikan checkpoint serta batch parsial sebelum menjalankan lagi.

`requested_convolution_units` dan tiap `prediction_events` mencatat compute route yang diminta, resource dan waktunya. Permintaan CPU+GPU tidak membuktikan penempatan GPU aktual. Bandingkan waktu convolution dan seluruh `pcm_pipeline_seconds`, load, thermal serta memory snapshots. Report baru memuat hash manifest dan repair agar aset terikat pada hasil. Snapshot memori bukan peak sebenarnya.

Bila GPU frontend lebih cepat dan kualitas/stabilitas tetap terjaga, lanjutkan pemeriksaan kualitas menyeluruh dengan route tersebut serta pengemasan bobot. Aset statis dan dinamis pada benchmark masih terduplikasi; ukurannya bukan ukuran paket produksi. Jangan mengubah routing SongMap berdasarkan satu run ini.
